# Verify MultiCamera HandRemoval

In [ ]:
experiment = "hand_removal"
run = "multi"
expruns_path = None
results_path = None
creation_style = "exist-ok"

In [ ]:
import sys
sys.path.insert(0, "../../..")
from pathlib import Path
from exp_run_config import Config
from _experimental._2026_11_aloha_without_leader.hand_removal import stages
from _experimental._2026_11_aloha_without_leader.hand_removal.clip import Clip
from IPython.display import Video, display, Image
import cv2
import numpy as np

if expruns_path:
    Config().set_exprun_path(expruns_path)
if results_path:
    Config().set_results_path(results_path)
exp = Config().get_experiment(experiment, run, creation_style=creation_style)
out = Path(exp["data_dir"])

def show(bgr):
    display(Image(data=cv2.imencode(".png", bgr)[1].tobytes()))

def preview(pattern):
    for path in sorted(out.glob(pattern)):
        print(path.relative_to(out))
        display(Video(str(path), embed=True))

In [ ]:
stages.verify(exp, multi=True)
preview("*/*/review.mp4")

Choose an episode/method directory, frame index and pixel (x, y) to inspect its fill source. Reference IDs index the manifest reference list; temporal and auxiliary IDs index the source video.

In [ ]:
def inspect_pixel(path, frame, x, y):
    with np.load(path/"provenance.npz") as p:
        labels = ["original","plate","reference","temporal","auxiliary","learned","unresolved"]
        return labels[p["source"][frame,y,x]], int(p["donor"][frame,y,x])

for review in sorted(out.glob("*/*/review.mp4"))[:1]:
    print(inspect_pixel(review.parent, 0, 40, 48))

Inspect a lossless frame at full resolution. Change the frame index or method to examine a difficult boundary.

In [ ]:
def inspect_frame(method_dir, clip, t):
    from _experimental._2026_11_aloha_without_leader.hand_removal.clip import still
    restored = still(method_dir/"frames"/f"{t:06}.png")
    panels = [clip.rgb[t], restored]
    if "clean" in clip.spec:
        truth = clip.series("clean")[t]
        panels.extend([truth, cv2.absdiff(truth, restored)])
    show(np.hstack(panels))
    print("Source frame:", clip.ids[t])

for clip in stages.clips(exp):
    methods = sorted((out/clip.spec["episode"]).glob("*/frames.json"))
    if methods:
        inspect_frame(methods[0].parent, clip, 0)
        break

In [ ]:
from _experimental._2026_11_aloha_without_leader.hand_removal.calibration import calibrate_pair, time_offset, triangulate_landmarks
# For a calibrated rig, supply matching checkerboard frames to calibrate_pair.
# For an existing clip, inspect source/target time differences and donor coverage.
import json
for table in sorted(out.glob("*/multiview/frames.json")):
    rows = json.loads(table.read_text())
    print(table.parent.parent.name, "auxiliary pixels per frame:", [r["auxiliary_pixels"] for r in rows])
    print("Largest auxiliary donor time difference (s):",
          max((r.get("max_donor_time_difference_s",0) for r in rows), default=0))

In [ ]:
exp.done()